In [1]:
import os 
import numpy as np
import matplotlib.colors as mcolors
# from functools import partial

os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = "false"
os.environ['jax_cpu'] = "cpu"

from vivarium.controllers.notebook_controller import NotebookController

In [2]:
controller = NotebookController()

In [3]:
def aggression(agent):
    return agent.right_prox, agent.left_prox

def fear(agent):
    return agent.left_prox, agent.right_prox

def love(agent):
    return 1. - agent.left_prox, 1. - agent.right_prox

def shy(agent):
    return 1. - agent.right_prox, 1. - agent.left_prox

In [4]:
for ag in controller.agents[:5]:
    ag.color = 'blue'
    ag.detach_all_behaviors()
    ag.attach_behavior(shy)
for ag in controller.agents[5:]:
    ag.color = 'purple'
    ag.detach_all_behaviors()
    ag.attach_behavior(aggression)

In [5]:
controller.run(threaded=True)

In [6]:
controller.stop()

In [7]:
# Could do a condition with the if 
def sensor_fn(dist, relative_theta, dist_max, cos_min, target_exist):
    cos_dir = np.cos(relative_theta)
    prox = 1. - (dist/dist_max)
    in_view = np.logical_and(dist < dist_max, cos_dir > cos_min)
    at_left = np.logical_and(True, np.sin(relative_theta) >= 0)
    left = in_view * at_left * prox
    right = in_view * (1. - at_left) * prox
    proxs = np.array([left, right]) * target_exist
    return proxs

In [15]:
def compute_agent_proxs(agent):
    agent_idx = agent.idx
    # raw proxs array of shape (n_entities - 1, 2) bc agent doesn't sense itself
    raw_proxs = np.zeros((len(controller.all_entities) - 1, 2))
    arr_idx = 0
    for sensed_ent_idx, entity in enumerate(controller.all_entities):
        if sensed_ent_idx != agent_idx:
            dist = agent.proximity_map_dist[sensed_ent_idx]
            theta = controller.agents[agent_idx].proximity_map_theta[sensed_ent_idx]
            dist_max = controller.agents[agent_idx].proxs_dist_max
            cos_min = controller.agents[agent_idx].proxs_cos_min
            target_exist = entity.exists

            sensors = sensor_fn(dist, theta, dist_max, cos_min, target_exist)
            raw_proxs[arr_idx] = sensors
            arr_idx += 1

    proxs = np.max(raw_proxs, axis=0)
    agent.left_prox = proxs[0]
    agent.right_prox = proxs[1]
    return proxs

In [17]:
def proxs_at_0_5(agent):
    # Set the proxs to 1 anyway to check the routine has an effect on the env
    agent.left_prox = 0.5
    agent.right_prox = 0.5

In [18]:
for ag in controller.agents:
    ag.attach_routine(proxs_at_0_5)

In [10]:
def check_agents_routines():
    for ag in controller.agents:
        routines = f"Agent {ag.idx} routines : "
        for routine in ag._routines:
            routines += str(routine) + " "
        print(routines)

In [11]:
def check_behaviors():
    for ag in controller.agents:
        behaviors = f"Agent {ag.idx} behaviors : "
        for behavior in ag.behaviors:
            behaviors += str(behavior) + " "
        print(behaviors)

In [12]:
# def compute_all_agents_proxs():
#     for agent in controller.agents:
#         proxs = compute_agent_proxs(agent)
#         left_prox, right_prox = proxs[0], proxs[1]
#         agent.left_prox = left_prox
#         agent.right_prox = right_prox

# compute_all_agents_proxs()

In [12]:
controller.run(threaded=True)

In [19]:
controller.stop()

In [20]:
for ag in controller.agents:
    ag.detach_all_routines()

In [21]:
check_agents_routines()

Agent 0 routines : 
Agent 1 routines : 
Agent 2 routines : 
Agent 3 routines : 
Agent 4 routines : 
Agent 5 routines : 
Agent 6 routines : 
Agent 7 routines : 
Agent 8 routines : 
Agent 9 routines : 


In [22]:
def rgb(color):
    return np.array(mcolors.to_rgb(color))

In [23]:
def selective_sensor(agent, sensed_color="red"):
    sensed_rgb_color = rgb(sensed_color)

    agent_idx = agent.idx
    entities = controller.all_entities
    # raw proxs array of shape (n_entities - 1, 2) bc agent doesn't sense itself
    raw_proxs = np.zeros((len(controller.all_entities), 2))
    for sensed_ent_idx, entity in enumerate(entities):
        if sensed_ent_idx != agent_idx:
            dist = agent.proximity_map_dist[sensed_ent_idx]
            theta = controller.agents[agent_idx].proximity_map_theta[sensed_ent_idx]
            dist_max = controller.agents[agent_idx].proxs_dist_max
            cos_min = controller.agents[agent_idx].proxs_cos_min
            target_exist = entity.exists

            sensors = sensor_fn(dist, theta, dist_max, cos_min, target_exist)
            raw_proxs[sensed_ent_idx] = sensors
        else:
            raw_proxs[sensed_ent_idx] = np.array([0., 0.])

    proxs = np.max(raw_proxs, axis=0)
    sensed_idx = np.argmax(raw_proxs, axis=0)

    # colors_sensed = np.array([rgb(entities[sensed_idx[0]].color), rgb(entities[sensed_idx[1]].color)])
    left_color_sened = rgb(entities[sensed_idx[0]].color)
    right_color_sened = rgb(entities[sensed_idx[1]].color)

    left_perceived = np.array_equal(left_color_sened, sensed_rgb_color)
    right_perceived = np.array_equal(right_color_sened, sensed_rgb_color)

    agent.left_prox = proxs[0] * left_perceived
    agent.right_prox = proxs[1] * right_perceived

In [24]:
def sense_reds(agent):
    return selective_sensor(agent, sensed_color="red")

def sense_purples(agent):
    return selective_sensor(agent, sensed_color="purple")

In [25]:
for ag in controller.agents[:5]:
    ag.color = 'green'
    ag.detach_all_behaviors()
    ag.detach_all_routines()
    ag.attach_behavior(aggression)
    ag.attach_routine(sense_purples)
for ag in controller.agents[5:]:
    ag.color = 'purple'
    ag.detach_all_behaviors()
    ag.detach_all_routines()
    ag.attach_behavior(aggression)
    ag.attach_routine(sense_reds)


In [26]:
check_agents_routines()

Agent 0 routines : sense_purples 
Agent 1 routines : sense_purples 
Agent 2 routines : sense_purples 
Agent 3 routines : sense_purples 
Agent 4 routines : sense_purples 
Agent 5 routines : sense_reds 
Agent 6 routines : sense_reds 
Agent 7 routines : sense_reds 
Agent 8 routines : sense_reds 
Agent 9 routines : sense_reds 


In [27]:
check_behaviors()

Agent 0 behaviors : aggression 
Agent 1 behaviors : aggression 
Agent 2 behaviors : aggression 
Agent 3 behaviors : aggression 
Agent 4 behaviors : aggression 
Agent 5 behaviors : aggression 
Agent 6 behaviors : aggression 
Agent 7 behaviors : aggression 
Agent 8 behaviors : aggression 
Agent 9 behaviors : aggression 


In [28]:
controller.run(threaded=True)

In [29]:
controller.stop()